In [1]:
!python -m pip install -q pandas numpy scikit-learn



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%%writefile code.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Iterable, Literal, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)

READMITTED_CLASSES_3: Tuple[str, str, str] = ("NO", "<30", ">30")


class TargetEngineeringError(ValueError):
    """Raised when the readmitted target contains unexpected/invalid values."""


def _normalize_readmitted_value(v: object) -> Optional[str]:
    """
    Normalize a single readmitted value:
    - None/NaN -> None
    - strings -> stripped, uppercased (except <30 and >30 remain as-is after upper)
    """
    if v is None:
        return None
    # pandas NA / numpy nan
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass

    s = str(v).strip()
    if s == "":
        return None
    return s.upper()


def validate_readmitted_values(
    values: Union[pd.Series, Iterable[object]],
    allowed: Sequence[str] = READMITTED_CLASSES_3,
    *,
    allow_na: bool = False,
) -> None:
    """
    Validate that all non-missing readmitted values are inside 'allowed'.

    Raises:
        TargetEngineeringError if an unexpected value is found.
    """
    allowed_set = {a.upper() for a in allowed}
    if isinstance(values, pd.Series):
        raw_iter = values.tolist()
    else:
        raw_iter = list(values)

    unexpected = set()
    for v in raw_iter:
        nv = _normalize_readmitted_value(v)
        if nv is None:
            if not allow_na:
                # Missing values are unexpected unless allow_na=True
                unexpected.add(None)
            continue
        if nv not in allowed_set:
            unexpected.add(nv)

    if unexpected:
        raise TargetEngineeringError(
            f"Unexpected readmitted values found: {sorted([x for x in unexpected if x is not None])}"
            + (" (and missing values)" if None in unexpected else "")
            + f". Allowed values: {list(allowed)}."
        )


def binarize_readmitted(
    readmitted: pd.Series,
    *,
    positive_value: str = "<30",
    negative_values: Sequence[str] = ("NO", ">30"),
    output_name: str = "readmitted_30d",
    unknown_policy: Literal["error", "nan"] = "error",
) -> pd.Series:
    """
    Convert {NO, <30, >30} into binary {1 if <30, 0 otherwise}.

    Args:
        readmitted: pandas Series with original readmitted values.
        positive_value: value mapped to 1.
        negative_values: values mapped to 0.
        output_name: name for the output series.
        unknown_policy:
            - "error": raise if any unexpected/NA values appear
            - "nan": map unexpected/NA to <NA> (nullable integer)

    Returns:
        pd.Series of dtype int8 (or nullable Int8 if unknown_policy="nan").
    """
    pos = positive_value.upper()
    neg = tuple(v.upper() for v in negative_values)

    # Normalize series
    norm = readmitted.map(_normalize_readmitted_value)

    mapping: Dict[str, int] = {pos: 1, **{v: 0 for v in neg}}

    if unknown_policy == "error":
        validate_readmitted_values(norm, allowed=(pos, *neg), allow_na=False)
        out = norm.map(mapping).astype(np.int8)
        out.name = output_name
        return out

    if unknown_policy == "nan":
        # allow NA/unknown -> <NA>
        out = norm.map(mapping)
        out = out.astype("Int8")  # nullable integer supports <NA>
        out.name = output_name
        return out

    raise ValueError(f"unknown_policy must be 'error' or 'nan', got: {unknown_policy}")


def keep_multiclass_readmitted(
    readmitted: pd.Series,
    *,
    output_name: str = "readmitted_3class",
    allowed: Sequence[str] = READMITTED_CLASSES_3,
    unknown_policy: Literal["error", "nan"] = "error",
) -> pd.Series:
    """
    Keep the original 3-class label, optionally validating values.

    Returns:
        pd.Series of dtype 'category' with categories in the allowed order,
        or with missing if unknown_policy="nan".
    """
    norm = readmitted.map(_normalize_readmitted_value)

    if unknown_policy == "error":
        validate_readmitted_values(norm, allowed=allowed, allow_na=False)
    elif unknown_policy == "nan":
        # allow NA/unknown; do not raise
        pass
    else:
        raise ValueError(f"unknown_policy must be 'error' or 'nan', got: {unknown_policy}")

    cat = pd.Categorical(norm, categories=[a.upper() for a in allowed], ordered=False)
    out = pd.Series(cat, index=readmitted.index, name=output_name)
    return out


def engineer_targets(
    df: pd.DataFrame,
    *,
    source_col: str = "readmitted",
    binary_col: str = "readmitted_30d",
    multiclass_col: str = "readmitted_3class",
    add_multiclass: bool = True,
    drop_source: bool = False,
    unknown_policy: Literal["error", "nan"] = "error",
) -> pd.DataFrame:
    """
    Add engineered target columns to a dataframe:
      - binary: 1 if <30 else 0
      - optional multiclass: categorical {NO, <30, >30}

    This keeps Section 2 self-contained and reproducible.

    Returns:
        A copy of df with new columns added (and optionally source_col dropped).
    """
    if source_col not in df.columns:
        raise KeyError(f"Column '{source_col}' not found in df. Available: {list(df.columns)}")

    out = df.copy()
    out[binary_col] = binarize_readmitted(
        out[source_col],
        output_name=binary_col,
        unknown_policy=unknown_policy,
    )

    if add_multiclass:
        out[multiclass_col] = keep_multiclass_readmitted(
            out[source_col],
            output_name=multiclass_col,
            unknown_policy=unknown_policy,
        )

    if drop_source:
        out.drop(columns=[source_col], inplace=True)

    return out


@dataclass(frozen=True)
class BinaryConfusionTerms:
    tp: int
    fp: int
    fn: int
    tn: int


def confusion_terms_binary(
    y_true: Union[pd.Series, np.ndarray, Sequence[int]],
    y_pred: Union[pd.Series, np.ndarray, Sequence[int]],
    *,
    positive_label: int = 1,
) -> BinaryConfusionTerms:
    """
    Return TP/FP/FN/TN for a binary task once the positive class is defined.

    This directly supports the Section 2 narrative about TP/FP/FN/TN meaning.
    """
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    # cm layout with labels [0,1] is:
    # [[TN, FP],
    #  [FN, TP]]
    tn, fp, fn, tp = cm.ravel()
    return BinaryConfusionTerms(tp=int(tp), fp=int(fp), fn=int(fn), tn=int(tn))


def metrics_binary(
    y_true: Union[pd.Series, np.ndarray, Sequence[int]],
    y_pred: Union[pd.Series, np.ndarray, Sequence[int]],
    y_score: Optional[Union[pd.Series, np.ndarray, Sequence[float]]] = None,
) -> Dict[str, Optional[float]]:
    """
    Convenience metric bundle for binary framing (Section 2 awareness):
      - accuracy
      - f1
      - balanced_accuracy
      - roc_auc (if y_score is provided)
    """
    res: Dict[str, Optional[float]] = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, pos_label=1)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "roc_auc": None,
    }
    if y_score is not None:
        res["roc_auc"] = float(roc_auc_score(y_true, y_score))
    return res


def metrics_multiclass(
    y_true: Union[pd.Series, np.ndarray, Sequence[str]],
    y_pred: Union[pd.Series, np.ndarray, Sequence[str]],
    y_proba: Optional[np.ndarray] = None,
    *,
    labels: Sequence[str] = READMITTED_CLASSES_3,
) -> Dict[str, Optional[float]]:
    """
    Metric bundle for the advanced extension (3-class framing):
      - macro_f1
      - weighted_f1
      - balanced_accuracy (macro recall)
      - ovr_auc_macro (if y_proba is provided with shape [n_samples, n_classes])

    Notes:
      - For AUC, we use one-vs-rest multi-class AUC (macro).
      - Requires y_proba columns correspond to 'labels' in the same order.
    """
    labels_up = [l.upper() for l in labels]

    yt = pd.Series(y_true).map(_normalize_readmitted_value)
    yp = pd.Series(y_pred).map(_normalize_readmitted_value)

    res: Dict[str, Optional[float]] = {
        "macro_f1": float(f1_score(yt, yp, labels=labels_up, average="macro")),
        "weighted_f1": float(f1_score(yt, yp, labels=labels_up, average="weighted")),
        "balanced_accuracy": float(balanced_accuracy_score(yt, yp)),
        "ovr_auc_macro": None,
    }

    if y_proba is not None:
        y_proba = np.asarray(y_proba)
        if y_proba.ndim != 2 or y_proba.shape[1] != len(labels_up):
            raise ValueError(
                f"y_proba must have shape [n_samples, {len(labels_up)}] matching labels order {labels_up}."
            )
        res["ovr_auc_macro"] = float(
            roc_auc_score(yt, y_proba, multi_class="ovr", average="macro", labels=labels_up)
        )

    return res


Writing code.py


In [1]:
%%writefile test_target_engineering.py
import os
import unittest
import importlib.util

import numpy as np
import pandas as pd

# Robust import of /content/.../code.py without colliding with the stdlib 'code' module
def import_module_from_path(module_name: str, file_path: str):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Cannot import module from {file_path}")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


PROJECT_DIR = os.path.dirname(os.path.abspath(__file__))
CODE_PATH = os.path.join(PROJECT_DIR, "code.py")
mod = import_module_from_path("project_code", CODE_PATH)


class TestTargetEngineering(unittest.TestCase):
    def test_binarize_basic_mapping(self):
        s = pd.Series(["NO", "<30", ">30", "NO", ">30", "<30"])
        y = mod.binarize_readmitted(s, unknown_policy="error")
        self.assertEqual(list(y.astype(int)), [0, 1, 0, 0, 0, 1])
        self.assertEqual(y.name, "readmitted_30d")
        self.assertTrue(str(y.dtype).lower() in ("int8", "int64", "int32"))

    def test_binarize_handles_whitespace_and_case(self):
        s = pd.Series([" no ", " <30", ">30 ", "No", "<30", " >30"])
        y = mod.binarize_readmitted(s, unknown_policy="error")
        self.assertEqual(list(y.astype(int)), [0, 1, 0, 0, 1, 0])

    def test_binarize_unknown_raises(self):
        s = pd.Series(["NO", "MAYBE", "<30"])
        with self.assertRaises(mod.TargetEngineeringError):
            _ = mod.binarize_readmitted(s, unknown_policy="error")

    def test_binarize_unknown_to_nan(self):
        s = pd.Series(["NO", "MAYBE", "<30", None])
        y = mod.binarize_readmitted(s, unknown_policy="nan")
        # Expect: NO->0, MAYBE->NA, <30->1, None->NA
        self.assertEqual(int(y.iloc[0]), 0)
        self.assertTrue(pd.isna(y.iloc[1]))
        self.assertEqual(int(y.iloc[2]), 1)
        self.assertTrue(pd.isna(y.iloc[3]))
        self.assertEqual(str(y.dtype), "Int8")

    def test_engineer_targets_adds_columns(self):
        df = pd.DataFrame(
            {
                "readmitted": ["NO", "<30", ">30"],
                "age": ["[50-60)", "[60-70)", "[40-50)"],
            }
        )
        out = mod.engineer_targets(df, add_multiclass=True, drop_source=False, unknown_policy="error")
        self.assertIn("readmitted_30d", out.columns)
        self.assertIn("readmitted_3class", out.columns)
        self.assertIn("readmitted", out.columns)
        self.assertEqual(list(out["readmitted_30d"].astype(int)), [0, 1, 0])
        # multiclass should be categorical with expected categories
        self.assertTrue(pd.api.types.is_categorical_dtype(out["readmitted_3class"]))

    def test_confusion_terms_binary(self):
        y_true = [1, 1, 0, 0, 1, 0]
        y_pred = [1, 0, 0, 1, 1, 0]
        terms = mod.confusion_terms_binary(y_true, y_pred)
        # Manually:
        # TP: positions 0 and 4 => 2
        # FN: position 1 => 1
        # FP: position 3 => 1
        # TN: positions 2 and 5 => 2
        self.assertEqual((terms.tp, terms.fn, terms.fp, terms.tn), (2, 1, 1, 2))

    def test_metrics_multiclass_with_auc(self):
        y_true = ["NO", "<30", ">30", "NO", "<30", ">30"]
        y_pred = ["NO", "<30", "NO", "NO", "<30", ">30"]
        # Probabilities aligned with labels order ("NO", "<30", ">30")
        y_proba = np.array(
            [
                [0.8, 0.1, 0.1],
                [0.1, 0.8, 0.1],
                [0.4, 0.2, 0.4],
                [0.7, 0.2, 0.1],
                [0.1, 0.7, 0.2],
                [0.2, 0.1, 0.7],
            ],
            dtype=float,
        )
        res = mod.metrics_multiclass(y_true, y_pred, y_proba=y_proba)
        for key in ["macro_f1", "weighted_f1", "balanced_accuracy", "ovr_auc_macro"]:
            self.assertIn(key, res)
            self.assertIsInstance(res[key], float)

    def test_keep_multiclass_error_policy(self):
        s = pd.Series(["NO", "<30", "???"])
        with self.assertRaises(mod.TargetEngineeringError):
            _ = mod.keep_multiclass_readmitted(s, unknown_policy="error")

    def test_keep_multiclass_nan_policy(self):
        s = pd.Series(["NO", "<30", "???"])
        out = mod.keep_multiclass_readmitted(s, unknown_policy="nan")
        self.assertTrue(pd.isna(out.iloc[2]))


if __name__ == "__main__":
    unittest.main(verbosity=2)


Overwriting test_target_engineering.py


In [7]:
!python -m unittest -v test_target_engineering.py


test_binarize_basic_mapping (test_target_engineering.TestTargetEngineering.test_binarize_basic_mapping) ... ok
test_binarize_handles_whitespace_and_case (test_target_engineering.TestTargetEngineering.test_binarize_handles_whitespace_and_case) ... ok
test_binarize_unknown_raises (test_target_engineering.TestTargetEngineering.test_binarize_unknown_raises) ... ok
test_binarize_unknown_to_nan (test_target_engineering.TestTargetEngineering.test_binarize_unknown_to_nan) ... ok
test_confusion_terms_binary (test_target_engineering.TestTargetEngineering.test_confusion_terms_binary) ... ok
test_engineer_targets_adds_columns (test_target_engineering.TestTargetEngineering.test_engineer_targets_adds_columns) ... ok
test_keep_multiclass_error_policy (test_target_engineering.TestTargetEngineering.test_keep_multiclass_error_policy) ... ok
test_keep_multiclass_nan_policy (test_target_engineering.TestTargetEngineering.test_keep_multiclass_nan_policy) ... ok
test_metrics_multiclass_with_auc (test_target_